In [ ]:
import os
import requests
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

sns.set_style("darkgrid")

In [ ]:
buoy_list = ["CCE1", "PAPA", "KEO", "JKEO", "WHOTS", "TAO110W", "TAO125W", "TAO140W", "TAO155W", "TAO170W", "TAO165E", "TAO8S165E", "STRATUS", "BTM", "ICELAND", "BOBOA", "SOFS", "GAKOA", "KODIAK", "SEAK", "M2", "CAPEELIZABETH", "CHABA", "CCE2", "NH10", "ALAWAI", "CHUUK", "CRIMP1", "CRIMP2", "KANEOHE", "KILONALU", "GRAYSREEF", "GULFOFMAINE", "CRESCENTREEF", "HOGREEF", "COASTALMS", "CHEECAROCKS", "LAPARGUERA"]

openwater_list = ["CCE1", "PAPA", "KEO", "JKEO", "WHOTS", "TAO110W", "TAO125W", "TAO140W", "TAO155W", "TAO170W", "TAO165E", "TAO8S165E", "STRATUS", "BTM", "ICELAND", "BOBOA", "SOFS"]
coastal_list = ["GAKOA", "KODIAK", "SEAK", "M2", "CAPEELIZABETH", "CHABA", "CCE2", "NH10", "GRAYSREEF", "GULFOFMAINE", "COASTALMS"]
coralreef_list = ["ALAWAI", "CHUUK", "CRIMP1", "CRIMP2", "KANEOHE", "KILONALU", "CRESCENTREEF", "HOGREEF", "CHEECAROCKS", "LAPARGUERA"]
polar_list = ["ICELAND"]
tropical_list = ["WHOTS", "TAO110W", "TAO125W", "TAO140W", "TAO155W", "TAO170W", "TAO165E", "TAO8S165E", "STRATUS", "BOBOA", "ALAWAI", "CHUUK", "CRIMP1", "CRIMP2", "KANEOHE", "KILONALU", "LAPARGUERA"]
temperate_list = ["CCE1", "PAPA", "KEO", "JKEO", "BTM", "SOFS", "GAKOA", "KODIAK", "SEAK", "M2", "CAPEELIZABETH", "CHABA", "CCE2", "NH10", "GRAYSREEF", "GULFOFMAINE", "CRESCENTREEF", "HOGREEF", "COASTALMS", "CHEECAROCKS"]

inputs_sst = ["SST"]
inputs_poly = ["SST", "year", "year_2", "week", "week_2", "week_3"]
output = "pCO2_sw"
date_range = None
focus_buoy = "WHOTS"
locations = ["coastal", "open water", "coral reef"]

In [ ]:
def download_file(url, filename):
    if os.path.exists(filename):
        return
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(filename, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
    except requests.RequestException as e:
        print(f"Failed: {filename} ({e})")

for buoy in buoy_list:
    download_file(f"https://www.pmel.noaa.gov/co2/timeseries/{buoy}.txt", f"{buoy}.txt")

In [ ]:
def classify(buoy):
    location = "open water" if buoy in openwater_list else "coastal" if buoy in coastal_list else "coral reef"
    climate = "temperate" if buoy in temperate_list else "tropical" if buoy in tropical_list else "polar"
    return location, climate

def add_features(df):
    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"])
    if date_range is not None:
        df = df[(df["datetime_utc"] >= date_range[0]) & (df["datetime_utc"] <= date_range[1])].copy()
    df["year"] = df["datetime_utc"].dt.year
    df["month"] = df["datetime_utc"].dt.month
    df["week"] = df["datetime_utc"].dt.isocalendar().week.astype(int)
    df["year_2"] = df["year"] ** 2
    df["week_2"] = df["week"] ** 2
    df["week_3"] = df["week"] ** 3
    if "CHL" in df.columns:
        df["CHL_log"] = np.log(df["CHL"].where(df["CHL"] > 0))
    return df

def load_buoy(buoy):
    df = pd.read_csv(f"{buoy}.txt", sep="\t", comment="#", na_values=[-999, -999.0])
    df["name"] = buoy
    df["location"], df["type"] = classify(buoy)
    return add_features(df)

buoys = {b: load_buoy(b) for b in buoy_list if os.path.exists(f"{b}.txt")}
all_df = pd.concat(buoys.values(), ignore_index=True)

lat_df = pd.DataFrame({"name": list(buoys)})
if os.path.exists("latitude_data - Sheet1.tsv"):
    loc_df = pd.read_csv("latitude_data - Sheet1.tsv", sep="\t")
    loc_df["lat"] = pd.to_numeric(loc_df["location"].astype(str).str.split(",").str[0], errors="coerce")
    loc_df["long"] = pd.to_numeric(loc_df["location"].astype(str).str.split(",").str[1], errors="coerce")
    lat_df = lat_df.merge(loc_df[["name", "lat", "long"]], on="name", how="left")
    all_df = all_df.merge(lat_df, on="name", how="left")

print(len(buoys), "buoys,", len(all_df), "rows")
all_df.groupby("name")[output].describe()

In [ ]:
corr_vars = [c for c in ["pCO2_sw", "SST", "SSS", "CHL_log", "DOXY", "pH_sw", "pCO2_air", "xCO2_air", "lat", "year", "month", "week"] if c in all_df.columns]

plt.figure(figsize=(12, 10))
sns.heatmap(all_df[corr_vars].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Matrix Across All Buoys", fontsize=25)
plt.show()

In [ ]:
rows = []
for buoy, df in buoys.items():
    row = {"name": buoy, "location": df["location"].iloc[0], "type": df["type"].iloc[0]}
    for v in corr_vars:
        if v not in (output, "lat"):
            row[f"r_{v}"] = df[output].corr(df[v]) if v in df.columns else np.nan
    rows.append(row)
corr_df = pd.DataFrame(rows).sort_values(["location", "name"]).reset_index(drop=True)
r_cols = [c for c in corr_df.columns if c.startswith("r_")]

plt.figure(figsize=(14, 14))
sns.heatmap(corr_df.set_index("name")[r_cols], annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation with pCO2 by Buoy", fontsize=25)
plt.xlabel("Variable", fontsize=18)
plt.ylabel("Buoy", fontsize=18)
plt.show()

corr_df.groupby("location")[r_cols].mean().round(3)

In [ ]:
palette = dict(zip(locations, sns.color_palette("deep", 3)))

plt.figure(figsize=(18, 8))
order = corr_df.sort_values("r_SST")
plt.bar(order["name"], order["r_SST"], color=order["location"].map(palette))
plt.axhline(0, color="black", linewidth=1)
plt.title("SST vs pCO2 Correlation at Each Buoy", fontsize=30)
plt.xlabel("Buoy", fontsize=20)
plt.ylabel("Pearson r", fontsize=20)
plt.xticks(rotation=60, ha="right", fontsize=12)
plt.yticks(fontsize=15)
plt.legend(handles=[Patch(color=c, label=l) for l, c in palette.items()], fontsize=16)
plt.show()

print(corr_df.groupby("type")[["r_SST"]].agg(["mean", "std", "count"]).round(3))

In [ ]:
sample = all_df.dropna(subset=["SST", output]).sample(min(20000, all_df[["SST", output]].dropna().shape[0]), random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(24, 7), sharey=True)
for ax, loc in zip(axes, locations):
    sub = sample[sample["location"] == loc]
    sns.scatterplot(data=sub, x="SST", y=output, hue="name", s=6, alpha=0.4, legend=False, ax=ax)
    if len(sub) > 1:
        ax.set_title(f"{loc.capitalize()} (r = {sub['SST'].corr(sub[output]):.2f})", fontsize=22)
    ax.set_xlabel("SST", fontsize=18)
    ax.set_ylabel("pCO2", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
def fit_model(df, inputs):
    data = df.dropna(subset=inputs + [output]).reset_index(drop=True)
    if len(data) < 10:
        return None
    X, y = data[inputs], data[output]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression().fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    return {
        "model": model,
        "data": data,
        "y_pred": model.predict(X),
        "r2": r2_score(y_test, y_test_pred),
        "r2_train": r2_score(y_train, model.predict(X_train)),
        "rmse": np.sqrt(mean_squared_error(y_test, y_test_pred)),
    }

def components(fit):
    d, c = fit["data"], dict(zip(inputs_poly, fit["model"].coef_))
    sst = d["SST"] * c["SST"]
    trend = d["year"] * c["year"] + d["year_2"] * c["year_2"]
    season = d["week"] * c["week"] + d["week_2"] * c["week_2"] + d["week_3"] * c["week_3"]
    return sst - sst.mean(), trend - trend.mean(), season - season.mean()

def yearly_cycle(fit):
    c = dict(zip(inputs_poly, fit["model"].coef_))
    weeks = np.arange(1, 53)
    cycle = weeks * c["week"] + weeks ** 2 * c["week_2"] + weeks ** 3 * c["week_3"]
    return weeks, cycle - cycle.mean()

def trend_line(data):
    first, last = data["year"].min(), data["year"].max()
    first_val = data.loc[data["year"] == first, output].mean()
    last_val = data.loc[data["year"] == last, output].mean()
    span = max(last - first, 1)
    rate = (last_val - first_val) / span
    return rate * (data["year"] - first) + first_val, rate

In [ ]:
f_sst = fit_model(buoys[focus_buoy], inputs_sst)
f_poly = fit_model(buoys[focus_buoy], inputs_poly)
d = f_poly["data"]

print(f"SST-only   r2 = {f_sst['r2']:.3f}  rmse = {f_sst['rmse']:.2f}")
print(f"Polynomial r2 = {f_poly['r2']:.3f}  rmse = {f_poly['rmse']:.2f}  r2_train = {f_poly['r2_train']:.3f}")
print(dict(zip(inputs_poly, f_poly["model"].coef_)), f_poly["model"].intercept_)

for fit, label in [(f_sst, "the SST-Only Model"), (f_poly, "the Polynomial Model")]:
    plt.figure(figsize=(15, 10))
    plt.plot(fit["data"]["datetime_utc"], fit["data"][output], label="pCO2 Actual Measurements")
    plt.plot(fit["data"]["datetime_utc"], fit["y_pred"], "r", label="pCO2 Predicted Measurements")
    plt.title(f"{focus_buoy} pCO2 Predictions Using {label}", fontsize=35)
    plt.xlabel("Year", fontsize=30)
    plt.ylabel("pCO2", fontsize=30)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)
    plt.legend(fontsize=22)
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
ax.plot(d["datetime_utc"], d[output], label="pCO2 Actual Measurements", color="blue")
ax.plot(d["datetime_utc"], f_poly["y_pred"], label="pCO2 Predictions Using the Polynomial Model", color="red")
ax.plot(f_sst["data"]["datetime_utc"], f_sst["y_pred"], label="pCO2 Predictions Using Only SST", color="green")
ax.set_title("The SST-Only Model vs. the Polynomial Model", fontsize=35)
ax.set_xlabel("Date", fontsize=30)
ax.set_ylabel("pCO2", fontsize=30)
ax.tick_params(labelsize=20)
ax.legend(fontsize=20)
plt.show()

res_sst = f_sst["data"][output] - f_sst["y_pred"]
res_poly = d[output] - f_poly["y_pred"]

plt.figure(figsize=(15, 10))
plt.plot(f_sst["data"]["datetime_utc"], res_sst, "bo-", alpha=0.7, label="Error Using Only SST")
plt.plot(d["datetime_utc"], res_poly, "ro-", alpha=0.7, label="Error Using the Polynomial Model")
plt.title("Error Comparison between the SST-Only and Polynomial Models", fontsize=30)
plt.xlabel("Date", fontsize=30)
plt.ylabel("pCO2 Error", fontsize=30)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.legend(fontsize=18)
plt.show()

plt.figure(figsize=(15, 10))
plt.hist(res_sst, bins=30, alpha=0.7, color="green", label="Error Using the SST-Only Model")
plt.hist(res_poly, bins=30, alpha=0.7, color="red", label="Error Using the Polynomial Model")
plt.title("Error Histogram Comparison", fontsize=35)
plt.xlabel("Error", fontsize=30)
plt.ylabel("Frequency", fontsize=30)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.legend(fontsize=20)
plt.show()

In [ ]:
sst_c, trend_c, season_c = components(f_poly)

plt.figure(figsize=(15, 10))
plt.plot(d["datetime_utc"], sst_c, label="SST")
plt.plot(d["datetime_utc"], trend_c, label="Trend", color="orange")
plt.plot(d["datetime_utc"], season_c, label="Seasonality", color="purple")
plt.title(f"{focus_buoy} Prediction Components", fontsize=35)
plt.xlabel("Year", fontsize=30)
plt.ylabel("pCO2", fontsize=30)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.legend(fontsize=22)
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(15, 15), sharex=True)
for ax, comp, name, color in zip(axes, [sst_c, trend_c, season_c], ["SST", "Trend", "Seasonality"], ["C0", "orange", "purple"]):
    ax.plot(d["datetime_utc"], comp, color=color)
    ax.set_title(f"{focus_buoy} {name} Prediction Component", fontsize=22)
    ax.set_ylabel("pCO2", fontsize=18)
axes[-1].set_xlabel("Year", fontsize=18)
plt.tight_layout()
plt.show()

print(pd.DataFrame({"SST": sst_c, "Trend": trend_c, "Seasonality": season_c, "pCO2": d[output]}).corr().round(3))

In [ ]:
results, fits = [], {}
for buoy, df in buoys.items():
    fs, fp = fit_model(df, inputs_sst), fit_model(df, inputs_poly)
    if fs is None or fp is None:
        print(f"No valid data: {buoy}")
        continue
    fits[buoy] = {"sst": fs, "poly": fp}
    _, rate = trend_line(fp["data"])
    loc, typ = classify(buoy)
    results.append({
        "name": buoy, "location": loc, "type": typ,
        "n": len(fp["data"]),
        "years": fp["data"]["year"].max() - fp["data"]["year"].min() + 1,
        "pco2_mean": fp["data"][output].mean(),
        "pco2_std": fp["data"][output].std(),
        "trend_per_year": rate,
        "r2_old": fs["r2"], "r2_new": fp["r2"],
        "rmse_old": fs["rmse"], "rmse_new": fp["rmse"],
    })

results_df = pd.DataFrame(results)
results_df["r2_improvement"] = results_df["r2_new"] - results_df["r2_old"]
results_df["rmse_improvement"] = results_df["rmse_old"] - results_df["rmse_new"]
results_df["rmse_improvement (%)"] = results_df["rmse_improvement"] / results_df["rmse_old"] * 100
results_df.round(3)

In [ ]:
summary_cols = ["r2_old", "r2_new", "r2_improvement", "rmse_old", "rmse_new", "rmse_improvement"]
print(results_df[summary_cols].mean().round(3).to_string(), "\n")
print(results_df.groupby("location")[summary_cols].mean().round(3), "\n")
print(results_df.groupby("type")[summary_cols].mean().round(3))

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20, 11))
width = 0.4
for i, loc in enumerate(locations):
    sub = results_df[results_df["location"] == loc]
    x = np.arange(len(sub))
    for row, (metric, c_old, c_new) in enumerate([("r2", "blue", "cyan"), ("rmse", "red", "orange")]):
        ax = axes[row, i]
        label = "R²" if metric == "r2" else "RMSE"
        ax.bar(x - width / 2, sub[f"{metric}_old"], width, label=f"SST-Only {label}", color=c_old)
        ax.bar(x + width / 2, sub[f"{metric}_new"], width, label=f"Polynomial {label}", color=c_new)
        ax.set_title(f"{label} Values - {loc.capitalize()}", fontsize=22)
        ax.set_xticks(x)
        ax.set_xticklabels(sub["name"], rotation=45, ha="right", fontsize=11)
        ax.set_ylabel(f"{label} Value", fontsize=16)
        ax.set_xlabel("Buoys", fontsize=16)
        ax.tick_params(axis="y", labelsize=14)
        ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
names = list(fits)
ncols = 4
nrows = int(np.ceil(len(names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(24, 4.5 * nrows))
for ax, buoy in zip(axes.flat, names):
    data = fits[buoy]["poly"]["data"]
    line, rate = trend_line(data)
    ax.plot(data["datetime_utc"], data[output], color="b", linewidth=0.8)
    ax.plot(data["datetime_utc"], line, color="r", linewidth=2)
    ax.set_title(f"{buoy} ({rate:+.2f} µatm/yr)", fontsize=16)
for ax in list(axes.flat)[len(names):]:
    ax.axis("off")
fig.suptitle("Trend Emergence of pCO2 at Each Buoy", fontsize=30, y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7), sharey=True)
for ax, loc in zip(axes, locations):
    for buoy in results_df.loc[results_df["location"] == loc, "name"]:
        weeks, cycle = yearly_cycle(fits[buoy]["poly"])
        ax.plot(weeks, cycle, label=buoy)
    ax.set_title(f"pCO2 Yearly Cycle - {loc.capitalize()}", fontsize=22)
    ax.set_xlabel("Week", fontsize=18)
    ax.set_ylabel("pCO2", fontsize=18)
    ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

weeks, cycle = yearly_cycle(fits[focus_buoy]["poly"])
plt.figure(figsize=(15, 10))
plt.plot(np.arange(1, 105), np.tile(cycle, 2), "*-")
plt.title(f"{focus_buoy} Yearly pCO2 Cycle", fontsize=30)
plt.xlabel("Week", fontsize=22)
plt.ylabel("pCO2", fontsize=22)
plt.show()

results_df["season_amplitude"] = results_df["name"].map(lambda b: np.ptp(yearly_cycle(fits[b]["poly"])[1]))

In [ ]:
perf_df = results_df.merge(corr_df[["name"] + r_cols], on="name").merge(lat_df, on="name", how="left")
if "lat" in perf_df.columns:
    perf_df["abs_lat"] = perf_df["lat"].abs()

perf_vars = [c for c in ["r2_old", "r2_new", "r2_improvement", "rmse_new", "rmse_improvement", "pco2_std", "trend_per_year", "season_amplitude", "r_SST", "r_SSS", "r_CHL_log", "abs_lat", "years", "n"] if c in perf_df.columns]

plt.figure(figsize=(14, 12))
sns.heatmap(perf_df[perf_vars].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlations Between Model Performance and Buoy Characteristics", fontsize=22)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(24, 7))
pairs = [("r_SST", "r2_old"), ("pco2_std", "rmse_new"), ("season_amplitude", "r2_improvement")]
for ax, (xv, yv) in zip(axes, pairs):
    sns.scatterplot(data=perf_df, x=xv, y=yv, hue="location", palette=palette, s=120, ax=ax)
    for _, r in perf_df.iterrows():
        ax.annotate(r["name"], (r[xv], r[yv]), fontsize=8, xytext=(3, 3), textcoords="offset points")
    ax.set_title(f"{yv} vs {xv} (r = {perf_df[xv].corr(perf_df[yv]):.2f})", fontsize=18)
plt.tight_layout()
plt.show()